# Exploração inicial — restrições eólicas (janeiro de 2025)

Inspeção reproduzível do arquivo bruto do ONS, sem transformação ou preenchimento de valores ausentes. O recorte analítico desta etapa é o Ceará (`id_estado == 'CE'`).

In [1]:
from hashlib import sha256
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_FILE = PROJECT_ROOT / 'data' / 'raw' / 'RESTRICAO_COFF_EOLICA_2025_01.csv'

if not DATA_FILE.is_file():
    raise FileNotFoundError(f'Arquivo bruto não encontrado: {DATA_FILE}')

file_hash = sha256(DATA_FILE.read_bytes()).hexdigest().upper()
print(f'Arquivo: {DATA_FILE.name}')
print(f'Tamanho: {DATA_FILE.stat().st_size:,} bytes')
print(f'SHA-256: {file_hash}')

Arquivo: RESTRICAO_COFF_EOLICA_2025_01.csv
Tamanho: 36,763,398 bytes
SHA-256: 2DBE2C4232126CC03BE12CD96977B58EBAD8FD0E07E4431F618A4696D17C02AC


In [2]:
df = pd.read_csv(DATA_FILE, sep=';', encoding='utf-8', low_memory=False)
df['din_instante'] = pd.to_datetime(df['din_instante'], errors='coerce')

print(f'Dimensões: {df.shape[0]:,} linhas × {df.shape[1]} colunas')
df.info()
display(df.head())
display(df.describe(include='number').T)
display(df.dtypes.rename('tipo_inferido').to_frame())

Dimensões: 232,128 linhas × 24 colunas
<class 'pandas.DataFrame'>
RangeIndex: 232128 entries, 0 to 232127
Data columns (total 24 columns):
 #   Column                          Non-Null Count   Dtype         
---  ------                          --------------   -----         
 0   id_subsistema                   232128 non-null  str           
 1   nom_subsistema                  232128 non-null  str           
 2   id_estado                       232128 non-null  str           
 3   nom_estado                      232128 non-null  str           
 4   nom_usina                       232128 non-null  str           
 5   id_ons                          232128 non-null  str           
 6   ceg                             232128 non-null  str           
 7   din_instante                    232128 non-null  datetime64[us]
 8   val_geracao                     232128 non-null  float64       
 9   val_geracaolimitada             41483 non-null   float64       
 10  val_disponibilidade         

,id_subsistema,nom_subsistema,id_estado,nom_estado,nom_usina,id_ons,ceg,din_instante,val_geracao,val_geracaolimitada,...,cod_origemrestricao,dsc_restricao,id_pontoconexao,nom_pontoconexao,nom_agenteoperador,val_geracaonaorealizadaapurada,num_minutos_rel,num_minutos_cnf,num_minutos_ene,num_minutos_restricao
0,N,Norte,MA,Maranhão,Conj. Paulino Neves,CJU_MAPLN,-,2025-01-01 00:00:00,404.073,NaN,...,NaN,NaN,MAMR--500-A,MIRANDA II500kVA,SIMM,NaN,NaN,NaN,NaN,NaN
1,N,Norte,MA,Maranhão,Conj. Paulino Neves,CJU_MAPLN,-,2025-01-01 00:30:00,399.297,NaN,...,NaN,NaN,MAMR--500-A,MIRANDA II500kVA,SIMM,NaN,NaN,NaN,NaN,NaN
2,N,Norte,MA,Maranhão,Conj. Paulino Neves,CJU_MAPLN,-,2025-01-01 01:00:00,397.171,NaN,...,NaN,NaN,MAMR--500-A,MIRANDA II500kVA,SIMM,NaN,NaN,NaN,NaN,NaN
3,N,Norte,MA,Maranhão,Conj. Paulino Neves,CJU_MAPLN,-,2025-01-01 01:30:00,396.539,NaN,...,NaN,NaN,MAMR--500-A,MIRANDA II500kVA,SIMM,NaN,NaN,NaN,NaN,NaN
4,N,Norte,MA,Maranhão,Conj. Paulino Neves,CJU_MAPLN,-,2025-01-01 02:00:00,388.572,NaN,...,NaN,NaN,MAMR--500-A,MIRANDA II500kVA,SIMM,NaN,NaN,NaN,NaN,NaN


,count,mean,std,min,25%,50%,75%,max
val_geracao,232128.0,48.001647,67.920031,0.0,6.502,23.591,60.36700,846.818
val_geracaolimitada,41483.0,59.438968,83.536718,0.0,2.800,29.567,78.45800,861.500
val_disponibilidade,232128.0,163.086262,140.950588,0.0,69.300,120.000,210.86725,973.100
val_geracaoreferencia,232128.0,49.499417,68.482172,0.0,7.559,24.519,62.18600,888.963
val_geracaoreferenciafinal,14666.0,35.495507,61.923916,0.0,0.000,9.024,42.45575,581.861
dsc_restricao,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
val_geracaonaorealizadaapurada,41483.0,26.482951,48.329378,0.0,0.354,8.189,29.69600,720.669
num_minutos_rel,41483.0,10.046405,13.979037,0.0,0.000,0.000,30.00000,30.000
num_minutos_cnf,41483.0,2.819420,8.551771,0.0,0.000,0.000,0.00000,30.000
num_minutos_ene,41483.0,15.526360,14.678804,0.0,0.000,25.000,30.00000,30.000


,tipo_inferido
id_subsistema,str
nom_subsistema,str
id_estado,str
nom_estado,str
nom_usina,str
id_ons,str
ceg,str
din_instante,datetime64[us]
val_geracao,float64
val_geracaolimitada,float64


In [3]:
source_summary = pd.Series({
    'inicio': df['din_instante'].min(),
    'fim': df['din_instante'].max(),
    'datas_invalidas': int(df['din_instante'].isna().sum()),
    'estados_distintos': int(df['id_estado'].nunique(dropna=True)),
    'linhas_duplicadas_exatas': int(df.duplicated().sum()),
    'linhas_em_chaves_duplicadas': int(df.duplicated(['id_ons', 'din_instante'], keep=False).sum()),
})
display(source_summary.rename('valor').to_frame())

,valor
inicio,2025-01-01 00:00:00
fim,2025-01-31 23:30:00
datas_invalidas,0
estados_distintos,9
linhas_duplicadas_exatas,0
linhas_em_chaves_duplicadas,0


In [4]:
ce = df.loc[df['id_estado'].eq('CE')].copy()
expected_intervals = 31 * 48

ce_summary = pd.Series({
    'linhas': len(ce),
    'id_ons_distintos': ce['id_ons'].nunique(dropna=True),
    'nomes_distintos': ce['nom_usina'].nunique(dropna=True),
    'intervalos_distintos': ce['din_instante'].nunique(dropna=True),
    'intervalos_esperados_no_mes': expected_intervals,
    'inicio': ce['din_instante'].min(),
    'fim': ce['din_instante'].max(),
    'duplicadas_exatas': int(ce.duplicated().sum()),
    'linhas_em_chaves_duplicadas': int(ce.duplicated(['id_ons', 'din_instante'], keep=False).sum()),
})
display(ce_summary.rename('valor').to_frame())

,valor
linhas,28272
id_ons_distintos,19
nomes_distintos,19
intervalos_distintos,1488
intervalos_esperados_no_mes,1488
inicio,2025-01-01 00:00:00
fim,2025-01-31 23:30:00
duplicadas_exatas,0
linhas_em_chaves_duplicadas,0


In [5]:
coverage = (
    ce.groupby(['id_ons', 'nom_usina', 'ceg'], dropna=False)
    .agg(linhas=('din_instante', 'size'), inicio=('din_instante', 'min'), fim=('din_instante', 'max'))
    .reset_index()
)
display(coverage)

interval_steps = (
    ce.sort_values(['id_ons', 'din_instante'])
    .groupby('id_ons')['din_instante']
    .diff()
    .value_counts()
)
display(interval_steps.rename('ocorrencias').to_frame())

,id_ons,nom_usina,ceg,linhas,inicio,fim
0,CEECVA,Cataventos do Acaraú I,EOL.CV.CE.033756-0.01,1488,2025-01-01,2025-01-31 23:30:00
1,CEEMAU,Malhadinha 1,EOL.CV.CE.030841-2.01,1488,2025-01-01,2025-01-31 23:30:00
2,CEUCZ,Eol Icaraizinho,EOL.CV.CE.028699-0.01,1488,2025-01-01,2025-01-31 23:30:00
3,CEUFM,Praia Formosa,EOL.CV.CE.028631-1.01,1488,2025-01-01,2025-01-31 23:30:00
4,CJU_CEARD,Conj. Aracati II,-,1488,2025-01-01,2025-01-31 23:30:00
5,CJU_CEAUD,Conj. Acaraú II,-,1488,2025-01-01,2025-01-31 23:30:00
6,CJU_CECBS,Conj. Cacimbas,-,1488,2025-01-01,2025-01-31 23:30:00
7,CJU_CECHE,Conj. Pedra Cheirosa,-,1488,2025-01-01,2025-01-31 23:30:00
8,CJU_CEFSA,Conj. Faísa,-,1488,2025-01-01,2025-01-31 23:30:00
9,CJU_CEICR,Conj. Icaraí,-,1488,2025-01-01,2025-01-31 23:30:00


,ocorrencias
din_instante,
0 days 00:30:00,28253


In [6]:
restriction_summary = (
    ce.groupby(['cod_razaorestricao', 'cod_origemrestricao'], dropna=False)
    .agg(
        linhas=('id_ons', 'size'),
        referencia_final_preenchida=('val_geracaoreferenciafinal', 'count'),
        gnra_preenchida=('val_geracaonaorealizadaapurada', 'count'),
        minutos_minimos=('num_minutos_restricao', 'min'),
        minutos_maximos=('num_minutos_restricao', 'max'),
    )
    .reset_index()
)
display(restriction_summary)

,cod_razaorestricao,cod_origemrestricao,linhas,referencia_final_preenchida,gnra_preenchida,minutos_minimos,minutos_maximos
0,CNF,LOC,7,0,7,30.0,30.0
1,ENE,SIS,2913,0,2913,2.0,30.0
2,REL,SIS,2772,2772,2772,1.0,30.0
3,NaN,NaN,22580,0,0,NaN,NaN


In [7]:
nulls = pd.DataFrame({
    'nulos': ce.isna().sum(),
    'percentual_nulos': ce.isna().mean().mul(100).round(2),
}).sort_values(['nulos', 'percentual_nulos'], ascending=False)
display(nulls)

numeric_columns = ce.select_dtypes(include='number').columns
negative_counts = ce[numeric_columns].lt(0).sum().rename('valores_negativos')
display(negative_counts.to_frame())

,nulos,percentual_nulos
dsc_restricao,28272,100.00
val_geracaoreferenciafinal,25500,90.20
val_geracaolimitada,22580,79.87
cod_razaorestricao,22580,79.87
cod_origemrestricao,22580,79.87
val_geracaonaorealizadaapurada,22580,79.87
num_minutos_rel,22580,79.87
num_minutos_cnf,22580,79.87
num_minutos_ene,22580,79.87
num_minutos_restricao,22580,79.87


,valores_negativos
val_geracao,0
val_geracaolimitada,0
val_disponibilidade,0
val_geracaoreferencia,0
val_geracaoreferenciafinal,0
dsc_restricao,0
val_geracaonaorealizadaapurada,0
num_minutos_rel,0
num_minutos_cnf,0
num_minutos_ene,0


In [8]:
import numpy as np

restricted = ce.loc[ce['val_geracaonaorealizadaapurada'].notna()].copy()
calculated_gnra = (restricted['val_geracaoreferencia'] - restricted['val_geracao']).clip(lower=0)
matches = np.isclose(
    restricted['val_geracaonaorealizadaapurada'],
    calculated_gnra,
    atol=1e-6,
)

reconciliation = pd.Series({
    'linhas_com_gnra': len(restricted),
    'linhas_compativeis_com_max_referencia_menos_geracao': int(matches.sum()),
    'maior_erro_absoluto': float(
        (restricted['val_geracaonaorealizadaapurada'] - calculated_gnra).abs().max()
    ),
})
display(reconciliation.rename('valor').to_frame())

,valor
linhas_com_gnra,5.692000e+03
linhas_compativeis_com_max_referencia_menos_geracao,5.692000e+03
maior_erro_absoluto,2.842171e-14


## Conclusões verificadas nesta etapa

- O arquivo contém 232.128 linhas e 24 colunas, cobrindo de 2025-01-01 00:00 a 2025-01-31 23:30.
- O recorte do Ceará contém 28.272 linhas: 19 identificadores ONS × 1.488 intervalos de meia hora.
- Não foram encontradas duplicidades exatas nem na chave candidata `id_ons + din_instante`.
- Há 5.692 linhas com código de restrição: 2.913 ENE, 2.772 REL e 7 CNF; PAR não aparece no recorte.
- A referência final está preenchida nas 2.772 linhas REL, comportamento compatível com o dicionário oficial vigente.
- Quinze dos 19 identificadores representam conjuntos com `ceg = '-'`; não devem ser tratados como parques individuais nem vinculados à ANEEL por CEG.
- `dsc_restricao` está integralmente ausente no recorte do Ceará. Ausência não será convertida em texto ou zero.
- Nas 5.692 linhas preenchidas, a GNRa coincide numericamente com `max(val_geracaoreferencia - val_geracao, 0)` dentro da tolerância de ponto flutuante; os minutos não são reaplicados a esse valor na fonte.
- A unidade de `val_geracaonaorealizadaapurada` precisa ser confirmada antes de qualquer agregação energética.